In [ ]:
import geopandas as gpd
import folium
from shapely.ops import unary_union
from shapely import make_valid

In [ ]:
# Leitura dos shapefiles de uso do solo
gdf_2000 = gpd.read_file("./data/soil_use_2000.shp")
gdf_2010 = gpd.read_file("./data/soil_use_2010.shp")
gdf_2023 = gpd.read_file("./data/soil_use_2023.shp")

# Leitura do limite municipal de Jundiaí
limite = gpd.read_file("./data/green_jundiai/limite_municipal_jundiai.shp")

In [ ]:
# Reprojetar tudo para SIRGAS 2000 UTM 23S (operações geométricas em metros)
CRS_PROJ = "EPSG:31983"

gdf_2000 = gdf_2000.to_crs(CRS_PROJ)
gdf_2010 = gdf_2010.to_crs(CRS_PROJ)
gdf_2023 = gdf_2023.to_crs(CRS_PROJ)
limite   = limite.to_crs(CRS_PROJ)

In [ ]:
def urban_union(gdf):
    """Retorna a união de polígonos urbanos como uma única geometria válida."""
    urbano = gdf[gdf["soil_use"] == "urbano"].copy()
    urbano["geometry"] = urbano.geometry.apply(make_valid)
    return unary_union(urbano.geometry)

urban_2000 = urban_union(gdf_2000)
urban_2010 = urban_union(gdf_2010)
urban_2023 = urban_union(gdf_2023)

limite_geom = make_valid(unary_union(limite.geometry))

In [ ]:
# Expansão urbana: áreas que se tornaram urbanas no período
# Intersectado com o limite municipal para garantir recorte correto
exp_2000_2010 = make_valid(urban_2010.difference(urban_2000)).intersection(limite_geom)
exp_2010_2023 = make_valid(urban_2023.difference(urban_2010)).intersection(limite_geom)

# Converter para GeoDataFrame em WGS84 para o Folium
CRS_GEO = "EPSG:4326"

gdf_exp_2000_2010 = gpd.GeoDataFrame(geometry=[exp_2000_2010], crs=CRS_PROJ).to_crs(CRS_GEO)
gdf_exp_2010_2023 = gpd.GeoDataFrame(geometry=[exp_2010_2023], crs=CRS_PROJ).to_crs(CRS_GEO)
limite_geo        = limite.to_crs(CRS_GEO)

# Explode para múltiplos polígonos (evita MULTIPOLYGON no Folium)
gdf_exp_2000_2010 = gdf_exp_2000_2010.explode(index_parts=False, ignore_index=True)
gdf_exp_2010_2023 = gdf_exp_2010_2023.explode(index_parts=False, ignore_index=True)

print(f"Expansão 2000-2010: {len(gdf_exp_2000_2010)} polígonos")
print(f"Expansão 2010-2023: {len(gdf_exp_2010_2023)} polígonos")

In [ ]:
# Centro do mapa: centroide do limite municipal
centro = limite_geo.geometry.union_all().centroid
lat, lon = centro.y, centro.x

# Criar mapa Folium com base Carto Positron
m = folium.Map(
    location=[lat, lon],
    zoom_start=11,
    tiles="CartoDB positron",
    name="Carto Positron"
)

# Adicionar base ESRI Satellite
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="ESRI Satélite",
    overlay=False,
    control=True
).add_to(m)

In [ ]:
# Camada 1: Expansão urbana 2000-2010 (laranja)
folium.GeoJson(
    gdf_exp_2000_2010,
    name="Expansão Urbana 2000–2010",
    style_function=lambda _: {
        "fillColor": "#f59e0b",
        "color": "#b45309",
        "weight": 1,
        "fillOpacity": 0.6
    },
    tooltip="Expansão 2000–2010"
).add_to(m)

# Camada 2: Expansão urbana 2010-2023 (vermelho)
folium.GeoJson(
    gdf_exp_2010_2023,
    name="Expansão Urbana 2010–2023",
    style_function=lambda _: {
        "fillColor": "#ef4444",
        "color": "#991b1b",
        "weight": 1,
        "fillOpacity": 0.6
    },
    tooltip="Expansão 2010–2023"
).add_to(m)

# Limite municipal (contorno)
folium.GeoJson(
    limite_geo,
    name="Limite Municipal – Jundiaí",
    style_function=lambda _: {
        "fillColor": "none",
        "color": "#1e3a5f",
        "weight": 2,
        "fillOpacity": 0
    }
).add_to(m)

In [ ]:
# Legenda HTML
import branca

legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    z-index: 9999;
    background: white;
    border: 2px solid #ccc;
    border-radius: 6px;
    padding: 12px 16px;
    font-size: 13px;
    font-family: Arial, sans-serif;
    box-shadow: 2px 2px 8px rgba(0,0,0,0.25);
">
<b>Expansão da Mancha Urbana</b><br><br>
<i style="background:#f59e0b;width:14px;height:14px;display:inline-block;margin-right:6px;border:1px solid #b45309;"></i>2000 – 2010<br>
<i style="background:#ef4444;width:14px;height:14px;display:inline-block;margin-right:6px;border:1px solid #991b1b;"></i>2010 – 2023<br>
<i style="background:none;width:14px;height:14px;display:inline-block;margin-right:6px;border:2px solid #1e3a5f;"></i>Limite Municipal
</div>
"""

legend = branca.element.MacroElement()
legend._template = branca.element.Template(legend_html)
m.get_root().add_child(legend)

# Controle de camadas
folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
# Salvar como HTML
m.save("mapa_expansao_urbana_jundiai.html")
print("Mapa salvo em mapa_expansao_urbana_jundiai.html")